# mmWave Radar Sensing: Human-Only Physical Optics

This tutorial isolates the radar-to-human-to-radar signal in an empty scene. It compares full and incremental physical-optics updates using the face-centroid and parent-face far-field analytic integration modes.

The focus is API usage and the visible effect of configuration choices. Detailed visibility, phase-consistency, and runtime profiling belong in validation or benchmark workflows rather than this demo.

In [ ]:
# Resolve the local src/ tree and keep Matplotlib's cache outside the repository.
import os
import sys
import tempfile
import time

from pathlib import Path

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "src" / "mmWaveRadar").exists():
    repo_root = repo_root.parent
if not (repo_root / "src" / "mmWaveRadar").exists():
    raise RuntimeError(
        "Run this notebook from inside a HERMES source checkout, "
        "including the top-level demo/ directory."
    )
src_path = str(repo_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

os.environ.setdefault(
    "MPLCONFIGDIR",
    str(Path(tempfile.gettempdir()) / "mmwave-radar-mpl"),
)

import matplotlib.pyplot as plt
import numpy as np

from sionna.rt import load_scene

from mmWaveRadar import amass_to_smpl_motion_sequence
from mmWaveRadar.dsp import (
    plot_axis_image,
    range_doppler_map,
    range_fft,
    range_profile_from_cube,
)
from mmWaveRadar.materials import human_skin_material
from mmWaveRadar.radar import FMCWConfig, RadarHardware, RadarSensor
from mmWaveRadar.simulation import (
    HumanPOMobilityConfig,
    MmWaveRadarSimulator,
    db_relative,
    select_torch_device,
)
from mmWaveRadar.targets import MeshTarget
from mmWaveRadar.tutorial_support import (
    plot_mesh_projection,
    radar_pose_in_front_of_chest,
    time_window_mesh_sequence,
    plot_bedroom_scene_projection
)

## Radar Configuration

A single colocated Tx/Rx channel keeps the example compact. The FMCW transmitter count is derived from the hardware object so the two configurations cannot drift apart.

In [ ]:
hardware = RadarHardware.from_positions(
    tx_positions=[[0.0, 0.0, 0.0]],
    rx_positions=[[0.0, 0.0, 0.0]],
    name="single-virtual-channel",
)
fmcw = FMCWConfig(
    carrier_frequency=60e9,
    slope=68e12,
    chirp_duration=58e-6,
    chirp_repetition_time=65e-6,
    sampling_frequency=4.5e6,
    num_adc_samples=225,
    num_chirps_per_frame=64,
    frame_period=50e-3,
    num_tx=hardware.num_tx,
)

# One frame provides 64 chirps for the range-Doppler comparison.
num_frames = 1
simulation_duration_s = num_frames * fmcw.frame_period

print(f"Wavelength: {1e3 * fmcw.wavelength:.2f} mm")
print(
    f"Run length: {num_frames} frame x {fmcw.num_chirps_per_frame} chirps "
    f"({simulation_duration_s:.3f} s radar interval)"
)

## AMASS Motion Mesh

The repository includes the CMU walking motion in AMASS-style base-SMPL format. SMPL model files must be obtained separately and placed under models/smpl_models/, or supplied with MMWAVE_SMPL_MODEL_DIR. The selected source window is rebased to radar time zero.

In [ ]:
dataset_root = Path(
    os.environ.get("MMWAVE_DATASET_ROOT", repo_root / "data")
).expanduser()
amass_npz_path = Path(
    os.environ.get(
        "MMWAVE_AMASS_NPZ",
        dataset_root / "AMASS" / "walking_poses_cmu_105_02.npz",
    )
).expanduser()
smpl_model_dir = Path(
    os.environ.get("MMWAVE_SMPL_MODEL_DIR", repo_root / "models" / "smpl_models")
).expanduser()

missing_paths = [
    path for path in (amass_npz_path, smpl_model_dir) if not path.exists()
]
if missing_paths:
    missing = "\n".join(f"  - {path}" for path in missing_paths)
    raise FileNotFoundError(
        "Human-Only-PO.ipynb requires these AMASS/SMPL inputs:\n"
        f"{missing}\n"
        "Obtain the licensed SMPL model separately and set "
        "MMWAVE_SMPL_MODEL_DIR if needed."
    )

smpl_device = select_torch_device()
raw_mesh_sequence = amass_to_smpl_motion_sequence(
    str(amass_npz_path),
    str(smpl_model_dir),
    model_type="smpl",
    device=smpl_device,
)

# Start where subject 105 begins walking; change this for another motion.
motion_start_time_s = 3.0
mesh_preview_duration_s = 1.0
mesh_sequence = time_window_mesh_sequence(
    raw_mesh_sequence,
    start_time_s=motion_start_time_s,
    duration_s=max(simulation_duration_s, mesh_preview_duration_s),
)

radar_position, radar_orientation, chest_front_point, _ = (
    radar_pose_in_front_of_chest(mesh_sequence, clearance_m=0.50)
)
radar = RadarSensor(
    name="radar",
    position=tuple(float(value) for value in radar_position),
    orientation=tuple(float(value) for value in radar_orientation),
    hardware=hardware,
    fmcw=fmcw,
)
human_material = human_skin_material("human-po-skin")

last_chirp_time_s = fmcw.chirp_time(
    num_frames - 1, fmcw.num_chirps_per_frame - 1
)
if last_chirp_time_s > float(mesh_sequence.times[-1]) + 1e-9:
    raise ValueError("The selected mesh window does not cover the last radar chirp.")

print(f"Motion: {amass_npz_path}")
print(f"SMPL model: {smpl_model_dir} ({smpl_device})")
print(
    f"Mesh: {mesh_sequence.vertex_count} vertices, "
    f"{mesh_sequence.faces.shape[0]} faces"
)
print(f"Radar position: {np.array2string(np.asarray(radar.position), precision=3)}")

### Human Mesh Preview

The columns show the beginning, middle, and end of the selected preview window. All panels use common limits. The green marker appears only in the first column because it is the radar's initial aim point, not a tracked chest landmark.

In [ ]:
mesh_preview_times_s = np.linspace(
    float(mesh_sequence.times[0]),
    min(mesh_preview_duration_s, float(mesh_sequence.times[-1])),
    3,
)

fig, axes = plt.subplots(2, 3, figsize=(14, 7), constrained_layout=True)
for column, time_s in enumerate(mesh_preview_times_s):
    plot_mesh_projection(
        axes[0, column],
        mesh_sequence,
        time_s,
        radar.position,
        f"Human mesh x-y, t={time_s:.3f} s",
        axes=(0, 1),
    )
    plot_bedroom_scene_projection(axes[0, column], axes=(0, 1))

    plot_mesh_projection(
        axes[1, column],
        mesh_sequence,
        time_s,
        radar.position,
        f"Human mesh x-z, t={time_s:.3f} s",
        axes=(0, 2),
    )
    plot_bedroom_scene_projection(axes[1, column], axes=(0, 2))

plt.show()

## Full and Incremental PO

The four cases compare full reconstruction with incremental updates for two integration modes. The incremental thresholds below are tutorial settings: adjust them together to trade computation for approximation accuracy.

In [ ]:
def make_po_config(*, incremental, integration_mode):
    return HumanPOMobilityConfig(
        visibility_samples_per_face=4,
        visibility_fade_chirps=8,
        visibility_use_phase_center=True,
        adaptive_visibility_sampling=incremental,
        incremental_update=incremental,
        incremental_visibility_refresh_chirps=64,
        incremental_full_refresh_chirps=0,
        incremental_normal_threshold_deg=20.0,
        incremental_centroid_displacement_threshold_m=0.020,
        incremental_area_relative_threshold=0.30,
        po_integration_mode=integration_mode,
    )


case_specs = [
    {
        "key": "full_centroid",
        "label": "Full centroid",
        "incremental": False,
        "integration_mode": "face_centroid",
    },
    {
        "key": "incremental_centroid",
        "label": "Incremental centroid",
        "incremental": True,
        "integration_mode": "face_centroid",
    },
    {
        "key": "full_analytic",
        "label": "Full analytic",
        "incremental": False,
        "integration_mode": "parent_face_far_field_analytic",
    },
    {
        "key": "incremental_analytic",
        "label": "Incremental analytic",
        "incremental": True,
        "integration_mode": "parent_face_far_field_analytic",
    },
]


def run_po_case(spec):
    run_target = MeshTarget(
        name="amass_human_po",
        mesh_sequence=mesh_sequence,
        material=human_material,
    )
    simulator = MmWaveRadarSimulator(
        mobility_mode="human_only_po",
        compute_backend="auto",
        compute_precision="float32",
        progress=False,
        human_po_config=make_po_config(
            incremental=spec["incremental"],
            integration_mode=spec["integration_mode"],
        ),
    )
    started = time.perf_counter()
    cube = simulator.run(load_scene(), radar, [run_target], num_frames=num_frames)
    elapsed_s = time.perf_counter() - started
    print(f"{spec['label']}: {elapsed_s:.2f} s")
    return {**spec, "cube": cube, "wall_time_s": elapsed_s}


results = {spec["key"]: run_po_case(spec) for spec in case_specs}

## Optional Human-Only RT Reference

Set run_rt_reference to True to append a one-frame RT result for the same isolated human. This is a comparison signal, not ground truth for PO.

In [ ]:
run_rt_reference = False

if run_rt_reference:
    rt_target = MeshTarget(
        name="amass_human_rt",
        mesh_sequence=mesh_sequence,
        material=human_material,
    )
    rt_simulator = MmWaveRadarSimulator(
        mobility_mode="rt_retrace",
        coupling_mode="one_target_bounce",
        max_depth=2,
        samples_per_src=80_000,
        max_num_paths_per_src=80_000,
        diffuse_reflection=True,
        retrace_once_per_frame=True,
        compute_backend="auto",
        compute_precision="float32",
        progress=False,
        seed=42,
    )
    started = time.perf_counter()
    rt_cube = rt_simulator.run(
        load_scene(), radar, [rt_target], num_frames=num_frames
    )
    elapsed_s = time.perf_counter() - started
    results["rt"] = {
        "key": "rt",
        "label": "RT reference",
        "cube": rt_cube,
        "wall_time_s": elapsed_s,
    }
    print(f"RT reference: {elapsed_s:.2f} s")
else:
    print("Skipping optional RT reference.")

## Range and Range-Doppler Products

Each focused comparison uses one shared dB reference for its mean range profiles and first-frame range-Doppler maps. A final bar chart summarizes wall-clock time.

In [ ]:
range_limit_m = 4.0
display_floor_db = -80.0


def extract_products(cube):
    adc = np.asarray(cube.adc)
    chirps = adc.reshape((-1, adc.shape[-2], adc.shape[-1]))
    range_cube, profile_ranges = range_fft(
        chirps, fmcw=fmcw, window="hann", nfft_mult=4
    )
    profile = range_profile_from_cube(
        range_cube, combine_antennas="sum_power", combine_chirps="mean"
    )
    rd_cube, rd_ranges, velocities = range_doppler_map(
        adc[0],
        fmcw=fmcw,
        num_tx=hardware.num_tx,
        win_range="hann",
        win_doppler="hann",
    )
    rd_power = np.sum(np.abs(rd_cube) ** 2, axis=-1)
    return profile, profile_ranges, rd_power, rd_ranges, velocities


for result in results.values():
    result["products"] = extract_products(result["cube"])


def plot_comparison(result_keys, title):
    selected = [results[key] for key in result_keys]
    profile_ranges = selected[0]["products"][1]
    profile_mask = profile_ranges <= range_limit_m
    profiles_db = db_relative(np.stack([
        result["products"][0][profile_mask] for result in selected
    ]))
    rd_ranges = selected[0]["products"][3]
    rd_mask = rd_ranges <= range_limit_m
    rd_maps_db = db_relative(np.stack([
        result["products"][2][:, rd_mask] for result in selected
    ]))

    fig, axes = plt.subplots(
        1, len(selected) + 1,
        figsize=(5 * (len(selected) + 1), 4.2),
        constrained_layout=True,
    )
    for result, profile_db in zip(selected, profiles_db):
        axes[0].plot(
            profile_ranges[profile_mask], profile_db, label=result["label"]
        )
    axes[0].set(
        xlim=(0, range_limit_m),
        ylim=(display_floor_db, 3),
        xlabel="range [m]",
        ylabel="power [dB, shared reference]",
        title="Mean range profile",
    )
    axes[0].grid(True, alpha=0.3)
    axes[0].legend(fontsize=8)

    rd_axes = axes[1:]
    for ax, result, rd_map_db in zip(rd_axes, selected, rd_maps_db):
        image = plot_axis_image(
            rd_map_db,
            rd_ranges[rd_mask],
            result["products"][4],
            ax=ax,
            xlabel="range [m]",
            ylabel="velocity [m/s]",
            title=result["label"],
            cmap="magma",
            vmin=display_floor_db,
            vmax=0,
        )
    fig.colorbar(image, ax=rd_axes, label="power [dB, shared reference]")
    fig.suptitle(title)
    plt.show()


comparison_groups = [
    ("Face-centroid PO", ["full_centroid", "incremental_centroid"]),
    ("Parent-face analytic PO", ["full_analytic", "incremental_analytic"]),
]
if "rt" in results:
    comparison_groups.append(
        ("Full centroid PO and RT", ["full_centroid", "rt"])
    )

for title, result_keys in comparison_groups:
    plot_comparison(result_keys, title)

timing_labels = [result["label"] for result in results.values()]
timing_values = [result["wall_time_s"] for result in results.values()]
fig, ax = plt.subplots(figsize=(8, 3.8), constrained_layout=True)
ax.bar(timing_labels, timing_values, color="#4c78a8")
ax.set(title="Wall-clock time", ylabel="seconds")
ax.tick_params(axis="x", rotation=20)
ax.grid(True, axis="y", alpha=0.3)
plt.show()